In [ ]:
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro.distributions as dist

In [ ]:
def generate(n_time=1000, n_cols=100, seed=42):
    np.random.seed(seed)
    key = jax.random.PRNGKey(seed)
    
    # X_cov ~ Poisson(lambda=1.5)
    X_cov = np.random.poisson(lam=1.5, size=(n_time, n_cols - 2)).astype(np.float32)
    
    Y_dense = np.zeros(n_time, dtype=np.int32)
    Y_sparse = np.zeros(n_time, dtype=np.int32)
    
    Y_dense[0:2] = np.random.poisson(5, size=2)
    Y_sparse[0:2] = np.random.poisson(0.5, size=2)
    
    alpha_true = 0.5
    concentration = 1.0 / alpha_true
    
    beta_fixed_d = np.array([0.5, 0.2, 0.1]) 
    
    beta_shrink_d = np.zeros(99)
    beta_shrink_d[0] = 0.4          
    beta_shrink_d[1:5] = np.array([0.6, -0.5, 0.5, -0.6]) 
    
    beta_fixed_s = np.array([0.2, 0.1, 0.05])
    gamma_fixed_s = np.array([-1.0, 0.2, 0.1]) 
    
    beta_shrink_s = np.zeros(99)
    beta_shrink_s[0] = 0.5          
    beta_shrink_s[1:5] = np.array([-0.6, 0.4, -0.5, 0.6]) 
    
    gamma_shrink_s = np.zeros(99)
    gamma_shrink_s[0] = 0.6         
    gamma_shrink_s[1:5] = np.array([0.7, -0.5, 0.6, -0.7]) 

    for t in range(2, n_time):
        key, k1, k2 = jax.random.split(key, 3)
        
        yd_lag1 = np.log1p(Y_dense[t-1])
        yd_lag2 = np.log1p(Y_dense[t-2])
        ys_lag1 = np.log1p(Y_sparse[t-1])
        ys_lag2 = np.log1p(Y_sparse[t-2])
        
        x_cov_lag1 = np.log1p(X_cov[t-1])
        
        H_f_d = np.array([1.0, yd_lag1, yd_lag2])
        H_s_d = np.concatenate(([ys_lag1], x_cov_lag1))
        
        eta_d = np.dot(H_f_d, beta_fixed_d) + np.dot(H_s_d, beta_shrink_d)
        mu_d = np.exp(np.clip(eta_d, -15.0, 15.0))
        
        # Y_dense[t] ~ NegativeBinomial(mean = mu_d, dispersion = alpha_true)
        Y_dense[t] = dist.NegativeBinomial2(mean=mu_d, concentration=concentration).sample(k1)
        
        H_f_s = np.array([1.0, ys_lag1, ys_lag2])
        H_s_s = np.concatenate(([yd_lag1], x_cov_lag1))
        
        eta_mu_s = np.dot(H_f_s, beta_fixed_s) + np.dot(H_s_s, beta_shrink_s)
        eta_pi_s = np.dot(H_f_s, gamma_fixed_s) + np.dot(H_s_s, gamma_shrink_s)
        
        mu_s = np.exp(np.clip(eta_mu_s, -15.0, 15.0))
        pi_s = np.clip(jax.nn.sigmoid(eta_pi_s), 1e-6, 1.0 - 1e-6)
        
        # Y_sparse[t] ~ ZeroInflatedNegativeBinomial(mean = mu_s, gate = pi_s, dispersion = alpha_true)
        Y_sparse[t] = dist.ZeroInflatedNegativeBinomial2(
            mean=mu_s, concentration=concentration, gate=pi_s
        ).sample(k2)

    col_names = [f"Event_{i}" for i in range(1, n_cols + 1)]
    
    data_matrix = np.column_stack((Y_dense, Y_sparse, X_cov))
    
    df = pd.DataFrame(data_matrix, columns=col_names)
    df = df.astype(int)
    
    return df

df = generate(n_time=1000, n_cols=100)

csv_filename = "simulation_dataset.csv"
df.to_csv(csv_filename, index=False)